In [10]:
import speech_recognition as sr 
import os 
from pydub import AudioSegment
from pydub.silence import split_on_silence
from tqdm import tqdm
from faster_whisper import WhisperModel

c:\Users\robin\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
CURRENT_DIRECTORY = r'C:/Users/robin/OneDrive/Desktop/2_Career/1_Projects/2026_Quote_Extraction_from_Podcasts/Modern_Wisdom'
WAV_DIRECTORY = r'C:/Users/robin/OneDrive/Desktop/2_Career/1_Projects/2026_Quote_Extraction_from_Podcasts/Modern_Wisdom/WAV'
TXT_DIRECTORY = r'C:/Users/robin/OneDrive/Desktop/2_Career/1_Projects/2026_Quote_Extraction_from_Podcasts/Modern_Wisdom/TXT'

In [18]:
WAV_PATH = "C:/Users/robin/OneDrive/Desktop/2_Career/1_Projects/2026_Quote_Extraction_from_Podcasts/Modern_Wisdom/WAV/How UFC Star Ben Askren Cheated Death - #1116.wav"

In [13]:
model = WhisperModel("base", device="cpu", compute_type="int8")

In [14]:
def transcribe_wav_to_text(wav_path, txt_directory):
    '''
    Transcribe speech in a wav file into text and save it as a .txt file
    
    Args:
        wav_path: path to the .wav file to transcribe
        txt_directory: directory to store the transformed .txt files
    '''
    
    segments, info = model.transcribe(wav_path)
    
    full_text = " ".join(segment.text.strip() for segment in segments)
    
    filename = os.path.splitext(os.path.basename(wav_path))[0]
    txt_path = os.path.join(txt_directory, f"{filename}.txt")
    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write(full_text)

In [19]:
transcribe_wav_to_text(WAV_PATH, TXT_DIRECTORY)

In [24]:
for name in os.listdir(WAV_DIRECTORY):
    filename = os.path.splitext(name)[0]
    wav_path = os.path.join(WAV_DIRECTORY, f"{filename}.wav")
    transcribe_wav_to_text(wav_path, TXT_DIRECTORY)

### Version 2 with Speaker Diarization

In [28]:
import os
from tqdm import tqdm
from faster_whisper import WhisperModel
from resemblyzer import VoiceEncoder, preprocess_wav
from sklearn.cluster import AgglomerativeClustering

CURRENT_DIRECTORY = r'C:/Users/robin/OneDrive/Desktop/2_Career/1_Projects/2026_Quote_Extraction_from_Podcasts/Modern_Wisdom'
WAV_DIRECTORY = r'C:/Users/robin/OneDrive/Desktop/2_Career/1_Projects/2026_Quote_Extraction_from_Podcasts/Modern_Wisdom/WAV'
TXT_DIRECTORY = r'C:/Users/robin/OneDrive/Desktop/2_Career/1_Projects/2026_Quote_Extraction_from_Podcasts/Modern_Wisdom/TXT'

whisper_model = WhisperModel("base", device="cpu", compute_type="int8")
encoder = VoiceEncoder()  # loads bundled weights, no download/token needed


def get_speaker_turns(wav_path, num_speakers=2, window_sec=1.5):
    """
    Estimate who's speaking when, using local voice-fingerprint clustering.
    num_speakers=2 assumes host + one guest — adjust if an episode has more.
    """
    wav = preprocess_wav(wav_path)
    _, cont_embeds, wav_splits = encoder.embed_utterance(
        wav, return_partials=True, rate=1 / window_sec
    )
    clustering = AgglomerativeClustering(n_clusters=num_speakers).fit(cont_embeds)
    labels = clustering.labels_

    turns = []
    for i, split in enumerate(wav_splits):
        start = split.start / 16000  # resemblyzer works internally at 16kHz
        end = split.stop / 16000
        turns.append((start, end, f"SPEAKER_{labels[i]}"))
    return turns


def speaker_at(turns, timestamp):
    """Find which speaker was talking at a given point in time."""
    for start, end, speaker in turns:
        if start <= timestamp <= end:
            return speaker
    return "UNKNOWN"


def transcribe_wav_to_text(wav_path, txt_directory):
    """Transcribe a wav file and tag each line with an estimated speaker."""
    segments, info = whisper_model.transcribe(wav_path)
    turns = get_speaker_turns(wav_path)

    lines = [f"[{speaker_at(turns, seg.start)}] {seg.text.strip()}" for seg in segments]
    full_text = "\n".join(lines)

    filename = os.path.splitext(os.path.basename(wav_path))[0]
    txt_path = os.path.join(txt_directory, f"{filename}.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(full_text)


if __name__ == "__main__":
    os.makedirs(TXT_DIRECTORY, exist_ok=True)
    wav_files = [f for f in os.listdir(WAV_DIRECTORY) if f.lower().endswith(".wav")]

    for name in tqdm(wav_files, unit="episode"):
        wav_path = os.path.join(WAV_DIRECTORY, name)
        transcribe_wav_to_text(wav_path, TXT_DIRECTORY)

ModuleNotFoundError: No module named 'resemblyzer'